In [ ]:
"""
segment grains: convert binary volume into labeled volume
"""

import numpy as np
import sys
sys.path.append("../../src")

# from scipy.ndimage import label, distance_transform_edt, center_of_mass, binary_fill_holes
# from skimage.morphology import binary_dilation, binary_erosion, ball, h_maxima
# from skimage.segmentation import watershed
# from sklearn.cluster import DBSCAN
# from skimage import measure, filters
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
import process_ct as ct


sample = 'FWZ-1-1_1p5um'
threshold = 20000
vol = np.load(f'../../processed_data/binary/{sample}_binary_{threshold}.npy')

print(vol.shape)

slice_ind = 50

plt.imshow(vol[slice_ind], cmap="gray")

# for volumes that have already been thresholded, set binary = vol
binary = vol

In [ ]:
# # IF NOT ALREADY BINARY VOLUME, FIND THRESHOLD AND MAKE BINARY VOLUME; OTHERWISE, SKIP THIS AND NEXT CELLS 

# # make histogram to find initial threshold

# # plt.hist(vol[slice_ind].flatten(), bins=100, range=(0,50000))
# plt.hist(vol.flatten(), bins=100, range=(5000,50000))

In [ ]:
# # experiment with different thresholds
# threshold = 15000
# binary = (vol > threshold).astype(np.uint8)
# # plt.imshow((vol > threshold)[slice_ind], cmap="gray")
# plt.imshow(binary.max(axis=0), cmap="gray")

In [ ]:
import time
from scipy.ndimage import label, distance_transform_edt, binary_opening, binary_closing, binary_fill_holes, binary_dilation, binary_erosion, generate_binary_structure
from skimage.morphology import ball, h_maxima
from skimage.segmentation import watershed
from sklearn.cluster import DBSCAN

max_radius = 2
buffer = 10 # make bigger buffer on edges (when reducing volume) so that it doesn't interfere with curvature calculations of mesh

# reduce volume edges
binary, offsets = ct.reduce_volume(binary, np.maximum(2*max_radius, buffer))

print(offsets)

# Create a 3D structuring element
structure = generate_binary_structure(rank=3, connectivity=1)
print(structure)

# Perform morphological opening to remove small white spots
opened = binary_opening(binary, structure=structure).astype(np.uint8)

# Perform morphological closing to fill small holes
cleaned = binary_closing(opened, structure=structure).astype(np.uint8)

# plt.imshow(cleaned[slice_ind - offsets[0]], cmap="gray")
plt.imshow(cleaned.max(axis=0), cmap="gray")

print(cleaned.shape)

In [ ]:
# step 2
print("Eroding ...")
binary = binary_erosion(cleaned, structure=structure, iterations=max_radius)

# plt.imshow(binary[slice_ind - offsets[0]], cmap="gray")
plt.imshow(binary.max(axis=0), cmap="gray")

In [ ]:
# *** SET DBSCAN EPS VALUE ***
# *** USE EPS = 15 FOR POLISHED AND EPS = 10 FOR UNPOLISHED -- old comment***
dbscan_eps = 20

# step 3
print("Applying watershed algorithm ...")
# Compute the distance transform
s = time.time()
distance_transform = distance_transform_edt(binary)
print(time.time()-s)
s = time.time()
# Identify local maxima as markers
# h-maxima suppresses small peaks in the distance transform to ensure markers are distinct
local_maxi = h_maxima(distance_transform, h=1)  # Adjust `h` based on your data
print(time.time()-s)
s = time.time()
coordinates = np.argwhere(local_maxi)  # Extract coordinates of maxima
print(time.time()-s)
s = time.time()
# Cluster maxima using DBSCAN
# eps: maximum distance between points to be in the same cluster
# min_samples: minimum points in a cluster (set to 1 for single-linkage behavior)

# cluster with dbscan
db = DBSCAN(eps=dbscan_eps, min_samples=1).fit(coordinates)  # Adjust eps based on voxel spacing
print(time.time()-s)
s = time.time()
# Assign a unique label to each cluster
cluster_labels = db.labels_
unique_clusters = np.unique(cluster_labels)

# Create new markers
new_markers = np.zeros_like(local_maxi, dtype=int)
for cluster_id in unique_clusters:
    cluster_coords = coordinates[cluster_labels == cluster_id]
    centroid = np.mean(cluster_coords, axis=0).astype(int)  # Compute centroid
    new_markers[tuple(centroid)] = cluster_id + 1  # Assign unique label (starting from 1)

clustered_coords = np.argwhere(new_markers > 0)
x,y=clustered_coords.T[1:]
plt.scatter(y,x,s=0.5,color='red')
plt.imshow(binary.max(axis=0),cmap='gray')
plt.show()

print(time.time()-s)




In [ ]:
s = time.time()

# Apply the watershed algorithm
# Use the negative distance transform as the input to watershed
labeled = watershed(-distance_transform, new_markers, mask=binary)

print(time.time()-s)

In [ ]:
# reassign labels so that they are sequential
print("Relabeling ...")
labeled = ct._relabel_volume(labeled)
labeled = labeled.astype(np.uint16)

print(labeled.max())
print(len(np.unique(labeled)))

# step 4
print("Dilating back ...")
binary = labeled > 0 # is this necessary?
for i in range(max_radius):
    new_binary = binary_dilation(binary, structure=structure)
    added_pixels = np.logical_xor(new_binary,binary)
    coords = np.argwhere(added_pixels)
    binary = new_binary
    for j in range(np.shape(coords)[0]):
        z, x, y = coords[j]
        # assuming we're not on the edge !! because no grains should be on the edge !!
        if labeled[z-1:z+2,x-1:x+2,y-1:y+2].shape != (3,3,3):
            print('Error: grain on edge of volume')
        adjs = labeled[z-1:z+2,x-1:x+2,y-1:y+2][structure.astype(bool)]
        uniques, counts = np.unique(adjs,return_counts=True)
        # if all zeros, assign to 0
        if uniques.size == 1:
            labeled[z,x,y] = uniques[0]
        else: # the following if else can probably be reduced to just what's in the if
            if uniques[0] == 0:
                uniques = uniques[1:] # slice off the zero
                counts = counts[1:] # slice off the zero
            else:
                print('no zeros')
            # assign label that has most adjacent squares. If there is a tie between labels,
            # assign 0
            # not editing the binary volume if it's 0!!!
            # Should be handled by the above code when surrounded by all 0's
            # there should only be 1 or 2 labels in unique now
            # because I don't think there are triple grain collisions
            if uniques.size == 1: # have edited uniques since above, so different.
                # This is just to determine if there are multiple labels in the running
                labeled[z,x,y] = uniques[0]
            elif counts[0] == counts[1]:
                labeled[z,x,y] = 0
            else:
                labeled[z,x,y] = uniques[np.argmax(counts)]

# reduce volume again with same buffer (larger buffer so that doesn't interfere with curvature calculations on mesh)
print(f'buffer = {buffer}')
labeled_reduced, new_offsets = ct.reduce_volume(labeled, buffer)

print(labeled_reduced.max())
print(len(np.unique(labeled_reduced)))

In [ ]:
collapsed_labeled = labeled_reduced.max(axis=0)

# Assign random colors to the grains
unique_labels = np.unique(collapsed_labeled)
colored_slice = np.zeros((*collapsed_labeled.shape, 3), dtype=np.uint8)

np.random.seed(42)  # For reproducibility
label_colors = {label: np.random.randint(0, 255, size=3) for label in unique_labels if label != 0}

for label, color in label_colors.items():
    colored_slice[collapsed_labeled == label] = color

# Set the background to white
colored_slice[collapsed_labeled == 0] = [255, 255, 255]

plt.imshow(colored_slice)
plt.show()

In [ ]:
# scratch work
xlo = 300 # 0 # 0 # 100 # 100 # 100 # 200 # 0 # 150 # 250 # 400
xhi = 400 # 100 # 100 # 200 # 200 # 200 # 300 # 120 # 250 # 350 # 450
ylo = 600 # 600 # 700 # 700 # 800 # 550 # 800 # 900 # 900 # 900 # 950
yhi = 690 # 700 # 800 # 800 # 900 # 600 # 950 # 1000 # 1000 # 980 # 1000
zoomed_color = colored_slice[xlo:xhi,ylo:yhi]
zoomed_labeled = collapsed_labeled[xlo:xhi,ylo:yhi]
plt.imshow(zoomed_color)
plt.show()

uniques, counts = np.unique(zoomed_labeled, return_counts=True)
print(uniques[1:])
print(counts[1:])

In [ ]:
edited = ct.join_grains(labeled_reduced, [(35, 267, 336), (121, 1666, 971), (114, 250, 328)])

edited = labeled_reduced.copy()

np.save(f'../../processed_data/labeled/{sample}_labeled_{threshold}_2r.npy', edited)

edited_collapsed = edited.max(axis=0)

# Assign random colors to the grains
unique_labels = np.unique(edited_collapsed)
colored_slice = np.zeros((*edited_collapsed.shape, 3), dtype=np.uint8)

np.random.seed(42)  # For reproducibility
label_colors = {label: np.random.randint(0, 255, size=3) for label in unique_labels if label != 0}

for label, color in label_colors.items():
    colored_slice[edited_collapsed == label] = color

# Set the background to white
colored_slice[edited_collapsed == 0] = [255, 255, 255]

plt.imshow(colored_slice)
plt.show()

In [ ]:
print(edited.shape)